# 03a - Dummy Baseline
Referencni spodni latka pro nevyrovnany dataset (dropout ~24 %). Vysledky se ukladaji do `results/dummy_results.pkl` pro porovnani v `03c

In [5]:
import joblib
import warnings
import os
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

warnings.filterwarnings('ignore')

X_train_prep, X_test_prep, y_train, y_test = joblib.load('../data/processed/split_data.pkl')

print("Rozlozeni trid v y_train:")
print(y_train.value_counts(normalize=True).to_string())

Rozlozeni trid v y_train:
Dropout
0    0.764625
1    0.235375


## Definice dummy klasifikatoru
- **most_frequent** - vzdy predikuje majoritni tridu (0); odhalí, zda je accuracy nasich modelu vubec relevantni
- **stratified** - nahodne predikce zachovavajici rozlozeni trid; ferovoejsi spodni hranice pro ROC-AUC

In [6]:
dummy_models = {
    'Dummy (most_frequent)': DummyClassifier(strategy='most_frequent', random_state=42),
    'Dummy (stratified)':    DummyClassifier(strategy='stratified',    random_state=42),
}

dummy_results = {}

for name, model in dummy_models.items():
    model.fit(X_train_prep, y_train)

    y_pred = model.predict(X_test_prep)

    # most_frequent nema smysluplne predict_proba pro roc_auc -> fallback na 0.5
    try:
        y_proba = model.predict_proba(X_test_prep)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    except Exception:
        roc_auc = 0.5

    dummy_results[name] = {
        'model':    model,
        'accuracy': accuracy_score(y_test, y_pred),
        'roc_auc':  roc_auc,
        'report':   classification_report(y_test, y_pred),
        'y_pred':   y_pred,
    }

    print(f"{'=' * 50}")
    print(f"  {name}")
    print(f"{'=' * 50}")
    print(f"  Accuracy : {dummy_results[name]['accuracy']:.4f}")
    print(f"  ROC-AUC  : {dummy_results[name]['roc_auc']:.4f}")
    print(f"\n{dummy_results[name]['report']}")

  Dummy (most_frequent)
  Accuracy : 0.7645
  ROC-AUC  : 0.5000

              precision    recall  f1-score   support

           0       0.76      1.00      0.87      1529
           1       0.00      0.00      0.00       471

    accuracy                           0.76      2000
   macro avg       0.38      0.50      0.43      2000
weighted avg       0.58      0.76      0.66      2000

  Dummy (stratified)
  Accuracy : 0.6490
  ROC-AUC  : 0.5133

              precision    recall  f1-score   support

           0       0.77      0.77      0.77      1529
           1       0.26      0.26      0.26       471

    accuracy                           0.65      2000
   macro avg       0.51      0.51      0.51      2000
weighted avg       0.65      0.65      0.65      2000



In [7]:
os.makedirs('../results', exist_ok=True)
joblib.dump(dummy_results, '../results/dummy_results.pkl')
print("Dummy vysledky ulozeny do results/dummy_results.pkl")

Dummy vysledky ulozeny do results/dummy_results.pkl
